# Part A, threshold variant: linear models chosen step by step

Comparison notebook. The main one is `part_a_final.ipynb`, where every step keeps the best option.

Predict a 1–5 rating from review text on Wildberries (WB) and Amazon with classical linear models. Meta features (field flags, length, category) belong to part B and are not used here.

**Data.** `train` is split by product into fit (80%) and validation (20%), seed 0. Every option is fitted on fit and scored on validation. The test split is read once, in the last step. Offsets of the decision rule are tuned on out-of-fold predictions of fit, 3 folds by product.

**Metrics.** macro-F1 is the main one, MAE the second.

**Selection rule.** Each step changes one thing. Options are listed from the simplest to the most complex, and the step keeps the first option whose score is within 0.003 of the best. A more complex option has to win by more than 0.003 to be kept.

**Steps.**
0. Baseline.
1. Features: n-grams, text preprocessing, TF-IDF scheme.
2. Model families, each with its own regularization grid. Families within 0.01 of the best one become leaders.
3. Class weights and decision rule for every leader, for macro-F1.
4. Regularization of every leader at its chosen weights and rule, for macro-F1.
5. Class weights and decision rule for every leader, for MAE.
6. Regularization of every leader for MAE.
7. Ensemble of the best tuned leaders, for each metric.
8. Final choice for each metric: tuned leaders in a fixed order of complexity, then the ensemble.
9. Test.

Finished runs are stored in `results_threshold.csv` and are not refitted. Run from this folder after `python ml/prepare_splits.py`.

In [1]:
import functools
import itertools
import json
import pathlib
import re
import time

import numpy as np
import pandas as pd
import pymorphy3
import scipy.sparse as sp
from nltk.stem import snowball
from sklearn import base
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection
from sklearn import naive_bayes
from sklearn import svm
from sklearn.feature_extraction import text as sk_text

SPLITS = pathlib.Path('../../data/splits')
RESULTS = pathlib.Path('results_threshold.csv')
DATASETS = ['wb', 'amazon']
SEED = 0
VAL_SIZE = 0.2
CLASSES = np.arange(1, 6)
EMOJI = re.compile('[\U0001F300-\U0001FAFF☀-➿]')
LANG = {'wb': 'russian', 'amazon': 'english'}
TIE = 0.003
LEADER_GAP = 0.01
ENSEMBLE_SIZE = 3
BIAS_GRID = np.linspace(-3, 3, 61)
WEIGHT_POWER = {'balanced': 1.0, 'sqrt': 0.5, 'none': 0.0}
OBJECTIVES = {'bias': 'macro_f1', 'bias_mae': 'mae'}
BASE_FEATURES = {'text': 'full', 'word': [1, 2], 'char': None, 'min_df': 3,
                 'max_features': 200_000, 'sublinear': True, 'binary': False}
BASELINE = {'features': BASE_FEATURES, 'model': {'name': 'logreg', 'C': 2.0},
            'imbalance': 'balanced', 'decode': 'argmax'}
COLUMNS = ['dataset', 'step', 'label', 'macro_f1', 'mae', 'accuracy',
           *[f'f1_{c}' for c in CLASSES], 'fit_sec', 'key']
CHOSEN = {}

In [2]:
class Data:
    """Train and test of one platform; train is split into fit and val."""

    def __init__(self, name):
        train, test = [pd.read_csv(SPLITS / f'{name}_{part}.csv.gz',
                                   keep_default_na=False)
                       for part in ['train', 'test']]
        splitter = model_selection.GroupShuffleSplit(
            n_splits=1, test_size=VAL_SIZE, random_state=SEED)
        fit_idx, val_idx = next(splitter.split(train, groups=train.product_id))
        # Test rows go last; only the test step reads them.
        self.df = pd.concat([train, test], ignore_index=True)
        self.y = self.df.rating.astype(int).to_numpy()
        self.groups = self.df.product_id.to_numpy()
        self.parts = {'val': (fit_idx, val_idx),
                      'test': (np.arange(len(train)),
                               np.arange(len(train), len(self.df)))}


def normalize(s):
    """Lowercases, maps emoji and '!' to tokens, collapses digits."""
    s = s.lower().replace('ё', 'е')
    s = EMOJI.sub(lambda m: f' emoji{ord(m.group()):x} ', s)
    s = re.sub(r'!+', ' excl ', s)
    return re.sub(r'\d+', '0', s)


def map_words(texts, fn):
    """Applies fn to every word, computing it once per unique word."""
    tokens = texts.str.findall(r'\w+')
    vocab = {w: fn(w) for w in set(itertools.chain.from_iterable(tokens))}
    return tokens.map(lambda ts: ' '.join(vocab[w] for w in ts))


@functools.cache
def make_text(dataset, variant):
    """Returns the review text as is, normalized, stemmed or lemmatized."""
    full = DATA[dataset].df.full_text
    if variant == 'full':
        return full
    if variant == 'clean':
        return full.map(normalize)
    if variant == 'stem':
        stem = snowball.SnowballStemmer(LANG[dataset]).stem
        return map_words(make_text(dataset, 'clean'), stem)
    morph = pymorphy3.MorphAnalyzer()
    return map_words(make_text(dataset, 'clean'),
                     lambda w: morph.parse(w)[0].normal_form)


DATA = {name: Data(name) for name in DATASETS}
pd.DataFrame({name: {part: len(idx[1]) for part, idx in d.parts.items()}
              | {'fit': len(d.parts['val'][0])}
              for name, d in DATA.items()})

,wb,amazon
val,50138,46766
test,58127,60736
fit,191457,192532


In [3]:
@functools.lru_cache(maxsize=2)
def features(dataset, spec_json, part):
    """Fits TF-IDF on the training side of a split, returns both sides."""
    spec = json.loads(spec_json)
    texts = make_text(dataset, spec['text'])
    tr, te = DATA[dataset].parts[part]
    common = dict(min_df=spec['min_df'], sublinear_tf=spec['sublinear'],
                  binary=spec['binary'], dtype=np.float32)
    vectorizers = []
    if spec['word']:
        vectorizers.append(sk_text.TfidfVectorizer(
            ngram_range=tuple(spec['word']),
            max_features=spec['max_features'], **common))
    if spec['char']:
        vectorizers.append(sk_text.TfidfVectorizer(
            analyzer='char_wb', ngram_range=tuple(spec['char']),
            max_features=300_000, **common))
    x_tr = sp.hstack([v.fit_transform(texts.iloc[tr]) for v in vectorizers])
    x_te = sp.hstack([v.transform(texts.iloc[te]) for v in vectorizers])
    return x_tr.tocsr(), x_te.tocsr()

In [4]:
class NBLogReg(base.BaseEstimator):
    """One-vs-rest logistic regressions on NB log-count-ratio features."""

    def __init__(self, C=1.0, alpha=1.0):
        self.C = C
        self.alpha = alpha

    def fit(self, x, y, sample_weight=None):
        """Fits one binary model per class."""
        self.classes_ = np.unique(y)
        self.ratios_, self.models_ = [], []
        for c in self.classes_:
            p = self.alpha + np.asarray(x[y == c].sum(axis=0)).ravel()
            q = self.alpha + np.asarray(x[y != c].sum(axis=0)).ravel()
            r = np.log((p / p.sum()) / (q / q.sum())).astype(np.float32)
            model = linear_model.LogisticRegression(C=self.C, max_iter=1000)
            model.fit(x.multiply(r).tocsr(), (y == c).astype(int),
                      sample_weight=sample_weight)
            self.ratios_.append(r)
            self.models_.append(model)
        return self

    def decision_function(self, x):
        """Returns per-class margins."""
        return np.column_stack([
            m.decision_function(x.multiply(r).tocsr())
            for m, r in zip(self.models_, self.ratios_)])

    def predict(self, x):
        """Returns the class with the largest margin."""
        return self.classes_[self.decision_function(x).argmax(axis=1)]


class Ensemble(base.BaseEstimator):
    """Mean of member scores scaled to unit spread; members keep weights."""

    def __init__(self, members=()):
        self.members = members

    def fit(self, x, y, sample_weight=None):
        """Fits every member with its own class-weight power."""
        self.models_, self.scales_ = [], []
        for spec in self.members:
            model = make_model({k: v for k, v in spec.items() if k != 'power'})
            model.fit(x, y, sample_weight=class_weights(y, spec['power']))
            self.models_.append(model)
            self.scales_.append(scores(model, x).std())
        return self

    def decision_function(self, x):
        """Returns the mean of scaled member scores."""
        return np.mean([scores(m, x) / s
                        for m, s in zip(self.models_, self.scales_)], axis=0)


MODELS = {
    'logreg': lambda **p: linear_model.LogisticRegression(max_iter=1000, **p),
    'linsvc': svm.LinearSVC,
    'ridge_clf': linear_model.RidgeClassifier,
    'nbsvm': NBLogReg,
    'sgd': lambda **p: linear_model.SGDClassifier(
        loss='log_loss', random_state=SEED, **p),
    'cnb': naive_bayes.ComplementNB,
    'mnb': naive_bayes.MultinomialNB,
    'ensemble': Ensemble,
}
# Models with probabilities, so the median rule applies to them.
PROBA = {'logreg', 'sgd', 'cnb', 'mnb'}


def make_model(spec):
    """Creates an unfitted model from {'name': ..., **params}."""
    params = {k: v for k, v in spec.items() if k != 'name'}
    return MODELS[spec['name']](**params)


def class_weights(y, power):
    """Returns balanced class weights raised to a power."""
    counts = pd.Series(y).value_counts()
    return ((len(y) / (len(CLASSES) * counts)).reindex(y).to_numpy())**power

In [5]:
def scores(model, x):
    """Continuous output: log-probabilities or margins."""
    if hasattr(model, 'predict_proba'):
        return np.log(np.clip(model.predict_proba(x), 1e-12, None))
    return model.decision_function(x)


def median_decode(log_proba):
    """Returns the median of the predicted distribution (MAE-optimal)."""
    cdf = np.cumsum(np.exp(log_proba), axis=1)
    return CLASSES[(cdf >= 0.5).argmax(axis=1)]


def tune_bias(s, y, objective):
    """Finds per-class score offsets maximizing macro-F1 or minimizing MAE."""
    def gain(offsets):
        pred = CLASSES[(s + offsets).argmax(1)]
        if objective == 'mae':
            return -metrics.mean_absolute_error(y, pred)
        return metrics.f1_score(y, pred, average='macro')

    bias = np.zeros(s.shape[1])
    for _, k in itertools.product(range(3), range(s.shape[1])):
        bias[k] = max(BIAS_GRID, key=lambda b: gain(
            np.r_[bias[:k], b, bias[k + 1:]]))
    return bias


def oof_scores(model, x, y, weight, groups):
    """Returns out-of-fold scores on the training side, 3 folds by product."""
    splitter = model_selection.GroupKFold(n_splits=3)
    folds = list(splitter.split(x, y, groups))
    parts = [
        scores(base.clone(model).fit(x[tr], y[tr], sample_weight=weight[tr]),
               x[te]) for tr, te in folds]
    order = np.concatenate([te for _, te in folds])
    return np.concatenate(parts)[np.argsort(order)]


def predict(cfg, x_tr, y_tr, groups, x_te):
    """Fits the model and decodes predictions with the configured rule."""
    model, decode = make_model(cfg['model']), cfg['decode']
    weight = class_weights(y_tr, WEIGHT_POWER[cfg['imbalance']])
    if decode in OBJECTIVES:
        offsets = tune_bias(oof_scores(model, x_tr, y_tr, weight, groups),
                            y_tr, OBJECTIVES[decode])
    model.fit(x_tr, y_tr, sample_weight=weight)
    if decode == 'argmax':
        return model.predict(x_te)
    s = scores(model, x_te)
    if decode == 'median':
        return median_decode(s)
    return CLASSES[(s + offsets).argmax(axis=1)]


def evaluate(y, pred):
    """Returns macro-F1, MAE, accuracy and per-class F1."""
    per_class = metrics.f1_score(
        y, pred, labels=CLASSES, average=None, zero_division=0)
    return {'macro_f1': metrics.f1_score(y, pred, average='macro'),
            'mae': metrics.mean_absolute_error(y, pred),
            'accuracy': metrics.accuracy_score(y, pred),
            **{f'f1_{c}': v for c, v in zip(CLASSES, per_class)}}

In [6]:
def key_of(cfg):
    """Returns the identity of a config: features, model, weights, rule."""
    return json.dumps(cfg, sort_keys=True)


def run(dataset, step_name, options):
    """Fits unlogged (label, config) options on validation, returns rows."""
    data = DATA[dataset]
    tr, te = data.parts['val']
    for label, cfg in options:
        results = pd.read_csv(RESULTS)
        if ((results.dataset == dataset) & (results.key == key_of(cfg))).any():
            continue
        spec = json.dumps(cfg['features'], sort_keys=True)
        x_tr, x_te = features(dataset, spec, 'val')
        start = time.time()
        pred = predict(cfg, x_tr, data.y[tr], data.groups[tr], x_te)
        row = {'dataset': dataset, 'step': step_name, 'label': label,
               **evaluate(data.y[te], pred),
               'fit_sec': round(time.time() - start, 1), 'key': key_of(cfg)}
        pd.DataFrame([row], columns=COLUMNS).to_csv(
            RESULTS, mode='a', header=False, index=False)
        print(dataset, step_name, label, round(row['macro_f1'], 4),
              round(row['mae'], 4), flush=True)
    results = pd.read_csv(RESULTS)
    rows = results[results.dataset == dataset].set_index('key')
    rows = rows.loc[[key_of(cfg) for _, cfg in options]].reset_index()
    return rows.assign(label=[label for label, _ in options])


def step(step_name, dataset, metric, options):
    """Runs options, keeps the first one within TIE of the best."""
    rows = run(dataset, step_name, options)
    sign = -1 if metric == 'mae' else 1
    score = sign * rows[metric].to_numpy()
    chosen = int(np.flatnonzero(score >= score.max() - TIE)[0])
    CHOSEN[step_name, dataset] = options[chosen][1]
    return rows.assign(step=step_name, dataset=dataset,
                       chosen=np.where(np.arange(len(rows)) == chosen, '✓', ''))


def show(tables):
    """Puts step tables of both datasets side by side, in option order."""
    rows = pd.concat(tables)
    order = rows[['step', 'label']].drop_duplicates()
    table = rows.pivot_table(index=['step', 'label'], columns='dataset',
                             values=['macro_f1', 'mae', 'chosen'],
                             aggfunc='first')
    table = table.reindex(pd.MultiIndex.from_frame(order))
    return table[['macro_f1', 'mae', 'chosen']].round(4)


def changed(cfg, **feature_changes):
    """Returns a config with some feature settings changed."""
    return {**cfg, 'features': {**cfg['features'], **feature_changes}}


def model_cfg(dataset, name, key, value, imbalance, decode):
    """Returns a config on the chosen features of a dataset."""
    return {'features': FEATURES[dataset], 'model': {'name': name, key: value},
            'imbalance': imbalance, 'decode': decode}

## Step 0. Baseline
TF-IDF of words and bigrams, logistic regression C=2 with balanced class weights, argmax.

In [7]:
# The reference point for every later step.
show([step('0_baseline', d, 'macro_f1', [('baseline', BASELINE)])
      for d in DATASETS])

macro_f1             mae         chosen   
dataset               amazon      wb  amazon      wb amazon wb
step       label                                              
0_baseline baseline   0.5224  0.4574  0.3691  0.3713      ✓  ✓

## Step 1. Features
The baseline model is kept fixed while the text representation changes. Each sub-step starts from the choice of the previous one.

### 1a. N-grams

In [8]:
show([step('1a_ngrams', d, 'macro_f1', [
    # Single words.
    ('words', changed(BASELINE, word=[1, 1])),
    # Words and bigrams.
    ('words + bigrams', BASELINE),
    # Words, bigrams and trigrams.
    ('words + bigrams + trigrams', changed(BASELINE, word=[1, 3])),
    # Character 2-5 grams inside word boundaries.
    ('char 2-5', changed(BASELINE, word=None, char=[2, 5])),
    # Words, bigrams and character 2-5 grams.
    ('words + bigrams + char 2-5', changed(BASELINE, char=[2, 5])),
]) for d in DATASETS])

macro_f1             mae         chosen  \
dataset                                amazon      wb  amazon      wb amazon   
step      label                                                                
1a_ngrams words                        0.4920  0.4322  0.4493  0.4440          
          words + bigrams              0.5224  0.4574  0.3691  0.3713      ✓   
          words + bigrams + trigrams   0.5237  0.4581  0.3627  0.3670          
          char 2-5                     0.5006  0.4431  0.4342  0.4161          
          words + bigrams + char 2-5   0.5212  0.4595  0.3709  0.3609          

                                         
dataset                              wb  
step      label                          
1a_ngrams words                          
          words + bigrams             ✓  
          words + bigrams + trigrams     
          char 2-5                       
          words + bigrams + char 2-5

Bigrams add 0.025 / 0.030 over single words (WB / Amazon). Trigrams (+0.0007 / +0.0013) and character n-grams on top of words (+0.0021 / −0.0012) stay within 0.003, and character n-grams alone lose 0.014 / 0.022. Words and bigrams are kept.

### 1b. Text preprocessing

In [9]:
def preprocessing_options(dataset):
    """Returns the text variants on top of the chosen n-grams."""
    prev = CHOSEN['1a_ngrams', dataset]
    options = [
        # Text as written.
        ('raw text', prev),
        # Lowercase, ё, emoji and '!' as tokens, digits collapsed.
        ('normalized', changed(prev, text='clean')),
        # Snowball stems of the normalized text.
        ('stems', changed(prev, text='stem')),
    ]
    # Pymorphy lemmas exist only for Russian.
    lemmas = [('lemmas', changed(prev, text='lemma'))]
    return options + lemmas * (dataset == 'wb')


show([step('1b_text', d, 'macro_f1', preprocessing_options(d))
      for d in DATASETS])

macro_f1             mae         chosen   
dataset              amazon      wb  amazon      wb amazon wb
step    label                                                
1b_text raw text     0.5224  0.4574  0.3691  0.3713      ✓   
        normalized   0.5235  0.4604  0.3674  0.3673         ✓
        stems        0.5166  0.4603  0.3713  0.3663          
        lemmas          NaN  0.4592     NaN  0.3666    NaN

WB: normalized text wins by 0.0030, exactly at the threshold, so this choice is fragile. Stems and lemmas add nothing on top of it. Amazon: normalization adds 0.0011, within the threshold, and stems lose 0.006, so the raw text is kept.

### 1c. TF-IDF scheme

In [10]:
def tfidf_options(dataset):
    """Returns term-frequency and vocabulary variants."""
    prev = CHOSEN['1b_text', dataset]
    return [
        # Sublinear tf, 200k features, min_df=3.
        ('sublinear tf, 200k', prev),
        # Raw term counts.
        ('raw tf', changed(prev, sublinear=False)),
        # Presence only.
        ('binary tf', changed(prev, binary=True)),
        # Vocabulary cut to 50k.
        ('50k features', changed(prev, max_features=50_000)),
        # Full vocabulary with min_df=2.
        ('full vocabulary', changed(prev, max_features=None, min_df=2)),
    ]


show([step('1c_tfidf', d, 'macro_f1', tfidf_options(d)) for d in DATASETS])

macro_f1             mae         chosen   
dataset                       amazon      wb  amazon      wb amazon wb
step     label                                                        
1c_tfidf sublinear tf, 200k   0.5224  0.4604  0.3691  0.3673         ✓
         raw tf               0.5224  0.4604  0.3708  0.3674          
         binary tf            0.5206  0.4603  0.3707  0.3664          
         50k features         0.5117  0.4525  0.3902  0.3818          
         full vocabulary      0.5255  0.4615  0.3627  0.3608      ✓

In [11]:
# Features used by every later step.
FEATURES = {d: CHOSEN['1c_tfidf', d]['features'] for d in DATASETS}
pd.DataFrame(FEATURES)

,wb,amazon
text,clean,full
word,"[1, 2]","[1, 2]"
char,None,None
min_df,3,2
max_features,200000,None
sublinear,True,True
binary,False,False


WB keeps the default scheme: the full vocabulary adds only 0.0011. Amazon takes the full vocabulary with min_df=2, +0.0031, again at the threshold. Raw or binary term frequency change nothing, and a 50k vocabulary loses 0.008 / 0.011.

## Step 2. Model families
Every family gets its own regularization grid, from the strongest regularization to the weakest, with balanced class weights and argmax as in the baseline. Families within 0.01 of the best one become leaders: later steps can still change their order.

In [12]:
# Family: (parameter, grid from the strongest regularization).
MODEL_GRID = {
    # Logistic regression.
    'logreg': ('C', [0.25, 0.5, 1.0, 2.0, 4.0]),
    # Linear SVM.
    'linsvc': ('C', [0.01, 0.03, 0.1, 0.3]),
    # Ridge classifier.
    'ridge_clf': ('alpha', [30.0, 10.0, 3.0, 1.0]),
    # Logistic regression on naive Bayes ratios (NB-SVM).
    'nbsvm': ('C', [0.3, 1.0, 3.0]),
    # SGD with log loss.
    'sgd': ('alpha', [3e-5, 1e-5, 3e-6]),
    # Complement naive Bayes.
    'cnb': ('alpha', [1.0, 0.3, 0.1]),
    # Multinomial naive Bayes.
    'mnb': ('alpha', [0.3, 0.1, 0.03]),
}

family_tables = [
    step(f'2_{name}', d, 'macro_f1',
         [(f'{key}={v}', model_cfg(d, name, key, v, 'balanced', 'argmax'))
          for v in grid])
    for d, (name, (key, grid)) in itertools.product(
        DATASETS, MODEL_GRID.items())]
show(family_tables)

macro_f1             mae         chosen   
dataset                   amazon      wb  amazon      wb amazon wb
step        label                                                 
2_logreg    C=0.25        0.5253  0.4612  0.3985  0.3864         ✓
            C=0.5         0.5291  0.4632  0.3827  0.3751      ✓   
            C=1.0         0.5272  0.4632  0.3711  0.3697          
            C=2.0         0.5255  0.4604  0.3627  0.3673          
            C=4.0         0.5220  0.4556  0.3573  0.3667          
2_linsvc    C=0.01        0.5194  0.4610  0.3935  0.3766          
            C=0.03        0.5290  0.4655  0.3712  0.3628      ✓  ✓
            C=0.1         0.5294  0.4628  0.3577  0.3623          
            C=0.3         0.5215  0.4568  0.3586  0.3672          
2_ridge_clf alpha=30.0    0.5238  0.4651  0.3792  0.3663         ✓
            alpha=10.0    0.5269  0.4648  0.3642  0.3603      ✓   
            alpha=3.0     0.5226  0.4571  0.3612  0.3721          
            alpha=1.0     0.5059  0.4449  0.3739  0.3908          
2_nbsvm     C=0.3         0.5180  0.4643  0.4041  0.3707         ✓
            C=1.0         0.5212  0.4644  0.3841  0.3550      ✓   
            C=3.0         0.5205  0.4593  0.3702  0.3544          
2_sgd       alpha=3e-05   0.5210  0.4608  0.4051  0.3879          
            alpha=1e-05   0.5275  0.4639  0.3806  0.3745      ✓  ✓
            alpha=3e-06   0.5270  0.4593  0.3614  0.3693          
2_cnb       alpha=1.0     0.5008  0.4425  0.4220  0.4203      ✓  ✓
            alpha=0.3     0.4848  0.4342  0.4348  0.4273          
            alpha=0.1     0.4736  0.4271  0.4515  0.4322          
2_mnb       alpha=0.3     0.4991  0.4453  0.4413  0.4338      ✓  ✓
            alpha=0.1     0.4985  0.4435  0.4152  0.4177          
            alpha=0.03    0.4913  0.4403  0.4018  0.4058

In [13]:
# Best setting of every family and the leaders.
families = pd.concat(family_tables)
families = families[families.chosen == '✓'].assign(
    family=lambda f: f.step.str.removeprefix('2_'))
families['gap'] = (families.macro_f1
                   - families.groupby('dataset').macro_f1.transform('max'))
families['leader'] = families.gap > -LEADER_GAP
LEADERS = {d: families[(families.dataset == d) & families.leader]
           .sort_values('macro_f1', ascending=False).family.tolist()
           for d in DATASETS}
families.round(4).pivot(index='family', columns='dataset',
                        values=['label', 'macro_f1', 'gap', 'leader'])

label              macro_f1             gap         leader  \
dataset         amazon           wb   amazon      wb  amazon      wb amazon   
family                                                                        
cnb          alpha=1.0    alpha=1.0   0.5008  0.4425 -0.0283  -0.023  False   
linsvc          C=0.03       C=0.03    0.529  0.4655 -0.0002     0.0   True   
logreg           C=0.5       C=0.25   0.5291  0.4612     0.0 -0.0044   True   
mnb          alpha=0.3    alpha=0.3   0.4991  0.4453   -0.03 -0.0202  False   
nbsvm            C=1.0        C=0.3   0.5212  0.4643  -0.008 -0.0012   True   
ridge_clf   alpha=10.0   alpha=30.0   0.5269  0.4651 -0.0022 -0.0005   True   
sgd        alpha=1e-05  alpha=1e-05   0.5275  0.4639 -0.0017 -0.0016   True   

                  
dataset       wb  
family            
cnb        False  
linsvc      True  
logreg      True  
mnb        False  
nbsvm       True  
ridge_clf   True  
sgd         True

Five families are within 0.01 of the best one and become leaders: the linear SVM, ridge, NB-SVM, SGD and logistic regression. The spread between them is 0.004 on WB and 0.008 on Amazon, so the order is not settled yet. Both naive Bayes models are 0.020–0.030 behind and drop out. Inside every family the rule keeps the strongest regularization among equal settings.

## Steps 3–4. Tuning the leaders for macro-F1
Step 3 picks class weights and the decision rule together, at the parameter from step 2, because they interact: without weights argmax almost never predicts 2★, while offsets fix that on their own. Options go from the step 2 setting (balanced, argmax) to offsets without weights. Step 4 then reruns the regularization grid at the chosen pair.

In [14]:
# Weights and rule pairs, from the step 2 setting.
F1_PAIRS = [(w, r) for r in ['argmax', 'bias'] for w in WEIGHT_POWER]

show([step(f'3_{name}', d, 'macro_f1',
           [(f'{w}, {r}', {**CHOSEN[f'2_{name}', d], 'imbalance': w,
                           'decode': r}) for w, r in F1_PAIRS])
      for d in DATASETS for name in LEADERS[d]])

macro_f1             mae         chosen   
dataset                        amazon      wb  amazon      wb amazon wb
step        label                                                      
3_linsvc    balanced, argmax   0.5290  0.4655  0.3712  0.3628          
            sqrt, argmax       0.5041  0.4438  0.3255  0.3229          
            none, argmax       0.4167  0.3651  0.3927  0.3832          
            balanced, bias     0.5371  0.4689  0.3436  0.3388      ✓   
            sqrt, bias         0.5368  0.4735  0.3308  0.3207         ✓
            none, bias         0.5320  0.4689  0.3358  0.3355          
3_ridge_clf balanced, argmax   0.5269  0.4651  0.3642  0.3663         ✓
            sqrt, argmax       0.5067  0.4257  0.3202  0.3389          
            none, argmax       0.4233  0.3287  0.3869  0.4293          
            balanced, bias     0.5345  0.4678  0.3384  0.3473          
            sqrt, bias         0.5385  0.4668  0.3236  0.3358      ✓   
            none, bias         0.5321  0.4587  0.3318  0.3399          
3_nbsvm     balanced, argmax   0.5212  0.4643  0.3841  0.3707          
            sqrt, argmax       0.5273  0.4498  0.3054  0.3046          
            none, argmax       0.4700  0.3830  0.3372  0.3571          
            balanced, bias     0.5351  0.4756  0.3255  0.3099      ✓  ✓
            sqrt, bias         0.5360  0.4741  0.3118  0.3099          
            none, bias         0.5346  0.4676  0.3119  0.3071          
3_sgd       balanced, argmax   0.5275  0.4639  0.3806  0.3745          
            sqrt, argmax       0.5272  0.4614  0.3152  0.3114          
            none, argmax       0.4443  0.3913  0.3639  0.3558          
            balanced, bias     0.5399  0.4739  0.3334  0.3224      ✓   
            sqrt, bias         0.5418  0.4775  0.3265  0.3164         ✓
            none, bias         0.5373  0.4790  0.3271  0.3144          
3_logreg    balanced, argmax   0.5291  0.4612  0.3827  0.3864          
            sqrt, argmax       0.5325  0.4583  0.3133  0.3164          
            none, argmax       0.4717  0.3883  0.3435  0.3615          
            balanced, bias     0.5409  0.4739  0.3351  0.3326      ✓   
            sqrt, bias         0.5422  0.4788  0.3301  0.3235         ✓
            none, bias         0.5420  0.4785  0.3289  0.3256

In [15]:
# Regularization at the chosen weights and rule.
show([step(f'4_{name}', d, 'macro_f1',
           [(f'{MODEL_GRID[name][0]}={v}',
             {**CHOSEN[f'3_{name}', d],
              'model': {'name': name, MODEL_GRID[name][0]: v}})
            for v in MODEL_GRID[name][1]])
      for d in DATASETS for name in LEADERS[d]])

macro_f1             mae         chosen   
dataset                   amazon      wb  amazon      wb amazon wb
step        label                                                 
4_linsvc    C=0.01        0.5257  0.4623  0.3608  0.3431          
            C=0.03        0.5371  0.4735  0.3436  0.3207      ✓  ✓
            C=0.1         0.5350  0.4751  0.3314  0.3148          
            C=0.3         0.5252  0.4687  0.3354  0.3232          
4_ridge_clf alpha=30.0    0.5328  0.4651  0.3348  0.3663         ✓
            alpha=10.0    0.5385  0.4648  0.3236  0.3603      ✓   
            alpha=3.0     0.5348  0.4571  0.3240  0.3721          
            alpha=1.0     0.5183  0.4449  0.3454  0.3908          
4_nbsvm     C=0.3         0.5344  0.4756  0.3288  0.3099      ✓  ✓
            C=1.0         0.5351  0.4743  0.3255  0.3136          
            C=3.0         0.5328  0.4656  0.3239  0.3233          
4_sgd       alpha=3e-05   0.5331  0.4730  0.3489  0.3318          
            alpha=1e-05   0.5399  0.4775  0.3334  0.3164      ✓  ✓
            alpha=3e-06   0.5360  0.4744  0.3229  0.3135          
4_logreg    C=0.25        0.5357  0.4788  0.3539  0.3235         ✓
            C=0.5         0.5409  0.4802  0.3351  0.3158      ✓   
            C=1.0         0.5359  0.4804  0.3255  0.3084          
            C=2.0         0.5322  0.4721  0.3308  0.3149          
            C=4.0         0.5255  0.4670  0.3330  0.3205

Offsets beat argmax for every leader except ridge on WB, where they add only 0.0027. The best class weights differ by model (balanced, square root or none), which is why they are chosen per model. After both steps logistic regression and SGD lead: 0.4788 / 0.5409 and 0.4775 / 0.5399. NB-SVM, the linear SVM and ridge follow within 0.014 / 0.007.

## Steps 5–6. Tuning the leaders for MAE
The same two steps with MAE as the target. Rules: argmax, the median of the predicted distribution (only for models with probabilities) and offsets tuned to minimize MAE.

In [16]:
def mae_pairs(name):
    """Returns weights and MAE rule pairs, simplest rule first."""
    rules = ['argmax'] + ['median'] * (name in PROBA) + ['bias_mae']
    return [(w, r) for r in rules for w in WEIGHT_POWER]


show([step(f'5_{name}', d, 'mae',
           [(f'{w}, {r}', {**CHOSEN[f'2_{name}', d], 'imbalance': w,
                           'decode': r}) for w, r in mae_pairs(name)])
      for d in DATASETS for name in LEADERS[d]])

macro_f1             mae         chosen   
dataset                          amazon      wb  amazon      wb amazon wb
step        label                                                        
5_linsvc    balanced, argmax     0.5290  0.4655  0.3712  0.3628          
            sqrt, argmax         0.5041  0.4438  0.3255  0.3229          
            none, argmax         0.4167  0.3651  0.3927  0.3832          
            balanced, bias_mae   0.5320  0.4643  0.3309  0.3254          
            sqrt, bias_mae       0.5361  0.4597  0.3197  0.3121          
            none, bias_mae       0.5169  0.4488  0.3156  0.3048      ✓  ✓
5_ridge_clf balanced, argmax     0.5269  0.4651  0.3642  0.3663          
            sqrt, argmax         0.5067  0.4257  0.3202  0.3389          
            none, argmax         0.4233  0.3287  0.3869  0.4293          
            balanced, bias_mae   0.5305  0.4548  0.3255  0.3308          
            sqrt, bias_mae       0.5240  0.4596  0.3145  0.3206      ✓  ✓
            none, bias_mae       0.5208  0.4426  0.3126  0.3205          
5_nbsvm     balanced, argmax     0.5212  0.4643  0.3841  0.3707          
            sqrt, argmax         0.5273  0.4498  0.3054  0.3046          
            none, argmax         0.4700  0.3830  0.3372  0.3571          
            balanced, bias_mae   0.5288  0.4554  0.3126  0.3021          
            sqrt, bias_mae       0.5318  0.4544  0.3045  0.2989      ✓  ✓
            none, bias_mae       0.5276  0.4482  0.3024  0.2995          
5_sgd       balanced, argmax     0.5275  0.4639  0.3806  0.3745          
            sqrt, argmax         0.5272  0.4614  0.3152  0.3114      ✓   
            none, argmax         0.4443  0.3913  0.3639  0.3558          
            balanced, median     0.4853  0.4216  0.4726  0.4430          
            sqrt, median         0.5274  0.4665  0.3300  0.3079         ✓
            none, median         0.5110  0.4345  0.3194  0.3146          
            balanced, bias_mae   0.5272  0.4570  0.3189  0.3154          
            sqrt, bias_mae       0.5371  0.4549  0.3153  0.3084          
            none, bias_mae       0.5241  0.4602  0.3157  0.3067          
5_logreg    balanced, argmax     0.5291  0.4612  0.3827  0.3864          
            sqrt, argmax         0.5325  0.4583  0.3133  0.3164          
            none, argmax         0.4717  0.3883  0.3435  0.3615          
            balanced, median     0.4972  0.4109  0.4501  0.4744          
            sqrt, median         0.5351  0.4586  0.3282  0.3222          
            none, median         0.5222  0.4272  0.3072  0.3173      ✓   
            balanced, bias_mae   0.5306  0.4687  0.3230  0.3229          
            sqrt, bias_mae       0.5381  0.4724  0.3140  0.3129          
            none, bias_mae       0.5399  0.4630  0.3108  0.3084         ✓

In [17]:
# Regularization at the chosen weights and MAE rule.
show([step(f'6_{name}', d, 'mae',
           [(f'{MODEL_GRID[name][0]}={v}',
             {**CHOSEN[f'5_{name}', d],
              'model': {'name': name, MODEL_GRID[name][0]: v}})
            for v in MODEL_GRID[name][1]])
      for d in DATASETS for name in LEADERS[d]])

macro_f1             mae         chosen   
dataset                   amazon      wb  amazon      wb amazon wb
step        label                                                 
6_linsvc    C=0.01        0.4995  0.4356  0.3440  0.3268          
            C=0.03        0.5169  0.4488  0.3156  0.3048          
            C=0.1         0.5259  0.4603  0.3044  0.2971      ✓  ✓
            C=0.3         0.5194  0.4495  0.3075  0.2999          
6_ridge_clf alpha=30.0    0.5147  0.4596  0.3360  0.3206          
            alpha=10.0    0.5240  0.4640  0.3145  0.3085      ✓  ✓
            alpha=3.0     0.5271  0.4563  0.3152  0.3072          
            alpha=1.0     0.5127  0.4502  0.3247  0.3212          
6_nbsvm     C=0.3         0.5280  0.4544  0.3140  0.2989          
            C=1.0         0.5318  0.4503  0.3045  0.2954      ✓  ✓
            C=3.0         0.5224  0.4508  0.3058  0.3030          
6_sgd       alpha=3e-05   0.4966  0.4347  0.3411  0.3339          
            alpha=1e-05   0.5272  0.4665  0.3152  0.3079          
            alpha=3e-06   0.5291  0.4763  0.3079  0.2966      ✓  ✓
6_logreg    C=0.25        0.5018  0.4630  0.3274  0.3084          
            C=0.5         0.5222  0.4696  0.3072  0.3011          
            C=1.0         0.5336  0.4669  0.2953  0.2956         ✓
            C=2.0         0.5375  0.4545  0.2913  0.2979      ✓   
            C=4.0         0.5377  0.4561  0.2923  0.3040

For MAE, plain argmax keeps its place only for SGD on Amazon. Everywhere else offsets tuned for MAE or the median win. After both steps the leaders reach 0.2954–0.3085 MAE on WB and 0.2913–0.3145 on Amazon. Logistic regression is the best on Amazon with the median (0.2913) and is 0.0002 behind NB-SVM on WB (0.2956 against 0.2954).

## Step 7. Ensemble
The best tuned leaders of a metric (at most three) are averaged. Each member keeps its own class weights, its scores are scaled to unit spread, and offsets for the target metric are tuned on the out-of-fold average.

In [18]:
def tuned(dataset, prefix):
    """Returns (family, config) of every tuned leader."""
    return [(name, CHOSEN[f'{prefix}_{name}', dataset])
            for name in LEADERS[dataset]]


def ensemble_cfg(dataset, prefix, metric, decode):
    """Returns an ensemble of the best tuned leaders for a metric."""
    options = tuned(dataset, prefix)
    rows = run(dataset, f'{prefix}_lookup', options)
    order = rows[metric].to_numpy().argsort()
    order = order if metric == 'mae' else order[::-1]
    members = [{**options[i][1]['model'],
                'power': WEIGHT_POWER[options[i][1]['imbalance']]}
               for i in order[:ENSEMBLE_SIZE]]
    return {'features': FEATURES[dataset],
            'model': {'name': 'ensemble', 'members': members},
            'imbalance': 'none', 'decode': decode}


show([step(f'7_ensemble_{metric}', d, metric,
           [(' + '.join(m['name'] for m in cfg['model']['members']), cfg)])
      for d in DATASETS
      for metric, prefix, decode in [('macro_f1', '4', 'bias'),
                                     ('mae', '6', 'bias_mae')]
      for cfg in [ensemble_cfg(d, prefix, metric, decode)]])

macro_f1             mae          \
dataset                                        amazon      wb  amazon      wb   
step                label                                                       
7_ensemble_macro_f1 logreg + sgd + nbsvm          NaN  0.4815     NaN  0.3121   
7_ensemble_mae      nbsvm + logreg + sgd          NaN  0.4663     NaN  0.2944   
7_ensemble_macro_f1 logreg + sgd + ridge_clf   0.5420     NaN  0.3226     NaN   
7_ensemble_mae      logreg + linsvc + nbsvm    0.5334     NaN  0.2987     NaN   

                                             chosen       
dataset                                      amazon   wb  
step                label                                 
7_ensemble_macro_f1 logreg + sgd + nbsvm        NaN    ✓  
7_ensemble_mae      nbsvm + logreg + sgd        NaN    ✓  
7_ensemble_macro_f1 logreg + sgd + ridge_clf      ✓  NaN  
7_ensemble_mae      logreg + linsvc + nbsvm       ✓  NaN

Ensembles of the three best tuned leaders reach 0.4815 / 0.5420 macro-F1 and 0.2944 / 0.2987 MAE. Over the best single model this is +0.0027 / +0.0011 macro-F1 and −0.0010 MAE on WB. On Amazon the MAE ensemble is worse than logistic regression alone.

## Step 8. Final choice
For each metric the tuned leaders are listed in a fixed order of complexity, then the ensemble. Logistic regression is the standard model with calibrated probabilities. Ridge and the linear SVM have no probabilities. SGD is a stochastic approximation. NB-SVM fits one model per class on reweighted features. The same rule keeps the first option within 0.003 of the best. Measured fit times are not used: they include pauses of the machine.

In [19]:
# Families from the simplest to the most complex.
COMPLEXITY = ['cnb', 'mnb', 'logreg', 'ridge_clf', 'linsvc', 'sgd', 'nbsvm']


def final_options(dataset, prefix, metric):
    """Returns tuned leaders by complexity, then the ensemble."""
    options = sorted(tuned(dataset, prefix),
                     key=lambda option: COMPLEXITY.index(option[0]))
    return options + [('ensemble', CHOSEN[f'7_ensemble_{metric}', dataset])]


show([step(f'8_final_{metric}', d, metric, final_options(d, prefix, metric))
      for d in DATASETS
      for metric, prefix in [('macro_f1', '4'), ('mae', '6')]])

macro_f1             mae         chosen   
dataset                      amazon      wb  amazon      wb amazon wb
step             label                                               
8_final_macro_f1 logreg      0.5409  0.4788  0.3351  0.3235      ✓  ✓
                 ridge_clf   0.5385  0.4651  0.3236  0.3663          
                 linsvc      0.5371  0.4735  0.3436  0.3207          
                 sgd         0.5399  0.4775  0.3334  0.3164          
                 nbsvm       0.5344  0.4756  0.3288  0.3099          
                 ensemble    0.5420  0.4815  0.3226  0.3121          
8_final_mae      logreg      0.5375  0.4669  0.2913  0.2956      ✓  ✓
                 ridge_clf   0.5240  0.4640  0.3145  0.3085          
                 linsvc      0.5259  0.4603  0.3044  0.2971          
                 sgd         0.5291  0.4763  0.3079  0.2966          
                 nbsvm       0.5318  0.4503  0.3045  0.2954          
                 ensemble    0.5334  0.4663  0.2987  0.2944

In [20]:
# Chosen configurations.
pd.DataFrame({(d, metric): {
    'model': CHOSEN[f'8_final_{metric}', d]['model'],
    'weights': CHOSEN[f'8_final_{metric}', d]['imbalance'],
    'rule': CHOSEN[f'8_final_{metric}', d]['decode'],
    'features': CHOSEN[f'8_final_{metric}', d]['features']}
    for d in DATASETS for metric in ['macro_f1', 'mae']})

wb  \
                                                   macro_f1   
model                         {'name': 'logreg', 'C': 0.25}   
weights                                                sqrt   
rule                                                   bias   
features  {'text': 'clean', 'word': [1, 2], 'char': None...   

                                                             \
                                                        mae   
model                          {'name': 'logreg', 'C': 1.0}   
weights                                                none   
rule                                               bias_mae   
features  {'text': 'clean', 'word': [1, 2], 'char': None...   

                                                     amazon  \
                                                   macro_f1   
model                          {'name': 'logreg', 'C': 0.5}   
weights                                            balanced   
rule                                                   bias   
features  {'text': 'full', 'word': [1, 2], 'char': None,...   

                                                             
                                                        mae  
model                          {'name': 'logreg', 'C': 2.0}  
weights                                                none  
rule                                                 median  
features  {'text': 'full', 'word': [1, 2], 'char': None,...

Logistic regression is chosen for both metrics on both datasets. For macro-F1 the ensemble is ahead by 0.0027 / 0.0011, within the threshold, so the simpler model wins. For MAE logistic regression is 0.0012 behind the ensemble on WB and the best on Amazon.

Final configurations:
- WB, macro-F1: C=0.25, square-root weights, offsets.
- WB, MAE: C=1, no weights, offsets for MAE.
- Amazon, macro-F1: C=0.5, balanced weights, offsets.
- Amazon, MAE: C=2, no weights, median.

## Step 9. Test
The chosen models are fitted on the whole train split and scored once on test. The baseline is shown for reference and was not used to choose anything.

In [21]:
def test_scores(dataset, cfg):
    """Fits on the whole train split, returns metrics on the test split."""
    data = DATA[dataset]
    tr, te = data.parts['test']
    spec = json.dumps(cfg['features'], sort_keys=True)
    x_tr, x_te = features(dataset, spec, 'test')
    pred = predict(cfg, x_tr, data.y[tr], data.groups[tr], x_te)
    return evaluate(data.y[te], pred)


pd.DataFrame({(d, label): test_scores(d, cfg)
              for d in DATASETS
              for label, cfg in [
                  ('baseline', BASELINE),
                  ('final for macro-F1', CHOSEN['8_final_macro_f1', d]),
                  ('final for MAE', CHOSEN['8_final_mae', d])]}).T.round(3)

macro_f1    mae  accuracy   f1_1   f1_2   f1_3  \
wb     baseline               0.457  0.368     0.764  0.549  0.164  0.290   
       final for macro-F1     0.482  0.314     0.796  0.596  0.165  0.326   
       final for MAE          0.464  0.292     0.815  0.607  0.048  0.370   
amazon baseline               0.520  0.382     0.711  0.708  0.286  0.357   
       final for macro-F1     0.540  0.343     0.745  0.710  0.325  0.370   
       final for MAE          0.531  0.309     0.765  0.720  0.290  0.369   

                            f1_4   f1_5  
wb     baseline            0.362  0.918  
       final for macro-F1  0.390  0.932  
       final for MAE       0.353  0.942  
amazon baseline            0.376  0.873  
       final for macro-F1  0.397  0.899  
       final for MAE       0.361  0.915

macro-F1 on test agrees with validation: 0.482 / 0.540 against 0.479 / 0.541. The MAE model gets 0.292 / 0.309 against 0.296 / 0.291 on validation, so on Amazon its test MAE is 0.018 worse. Against the baseline: +0.025 / +0.020 macro-F1 and −0.076 / −0.073 MAE. The MAE model gives up 2★: its F1 for 2★ on WB is 0.048.

The exploratory notebook `part_a.ipynb` took the maximum at every step and reached 0.484 / 0.546 and MAE 0.281 / 0.311 on test. The stricter rule costs at most 0.006 macro-F1 and 0.011 MAE, and every choice here has a stated reason.